# 🚀 Ollama GPU Server (Google Colab)

This notebook starts an Ollama server on Colab's free T4 GPU and exposes it
via ngrok for remote access by your AI TDD Orchestrator pipeline.

## Setup
1. Change runtime → **T4 GPU** (Runtime → Change runtime type → T4 GPU)
2. Set your ngrok auth token below (get one free at https://ngrok.com)
3. Run all cells
4. Copy the `COLAB_OLLAMA_URL` output and set it as env var

## Free Tier Limits
- GPU: NVIDIA T4 (15GB VRAM)
- Session: ~4-12 hours (reconnect if it dies)
- Unlimited sessions per day

In [ ]:
# ============================================
# CONFIGURATION — Set your ngrok auth token
# ============================================
NGROK_AUTH_TOKEN = ""  # Get free at https://dashboard.ngrok.com/get-started/your-authtoken
OLLAMA_MODEL = "qwen2.5-coder:7b"  # Model to pull

# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(f"🎮 GPU: {result.stdout.strip()}")
assert 'NVIDIA' in result.stdout, '❌ No GPU detected! Change runtime to T4 GPU.'

In [ ]:
# ============================================
# Step 1: Install Ollama
# ============================================
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama installed')

In [ ]:
# ============================================
# Step 2: Start Ollama server in background
# ============================================
import subprocess
import time
import requests

# Start Ollama server
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f'🔄 Ollama server started (PID: {proc.pid})')

# Wait for it to be ready
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print('✅ Ollama is ready!')
            break
    except:
        pass
    time.sleep(2)
else:
    print('❌ Ollama failed to start')

In [ ]:
# ============================================
# Step 3: Pull the model
# ============================================
!ollama pull {OLLAMA_MODEL}
print(f'✅ Model {OLLAMA_MODEL} ready')

In [ ]:
# ============================================
# Step 4: Expose via ngrok tunnel
# ============================================
!pip install pyngrok -q
from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print('\n' + '='*60)
print('🎉 OLLAMA IS LIVE!')
print('='*60)
print(f'\n📡 Public URL: {public_url}')
print(f'\n🔧 Set this environment variable:')
print(f'   export COLAB_OLLAMA_URL="{public_url}"')
print(f'\n🔧 Or add to GitHub Secrets:')
print(f'   COLAB_OLLAMA_URL = {public_url}')
print('\n' + '='*60)

In [ ]:
# ============================================
# Step 5: Keep alive (blocks cell to prevent session death)
# ============================================
import time

print('🔄 Keep-alive loop running. Do not close this tab.')
print('   Press ⏹️ to stop.')

uptime = 0
while True:
    time.sleep(60)
    uptime += 1
    # Ping Ollama to keep it warm
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
    except:
        pass
    if uptime % 10 == 0:
        print(f'⏱️  Uptime: {uptime} minutes | URL: {public_url}')